# Rerun PoET-2 on 4 different zero-shot tasks

## Define the directories and input

In [2]:
import pandas as pd
from os.path import exists
from os import chdir
import glob
import sys
import os
from scipy.stats import spearmanr,linregress
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import rankdata

chdir("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/")

dms_subs_ref_df = pd.read_csv("ProteinGym/reference_files/DMS_substitutions.csv")
dms_indels_ref_df = pd.read_csv("ProteinGym/reference_files/DMS_indels.csv")
clinical_subs_ref_df = pd.read_csv("ProteinGym/reference_files/clinical_substitutions.csv")
clinical_indels_ref_df = pd.read_csv("ProteinGym/reference_files/clinical_indels.csv")

dms_subs_output_dir = './PR/.cache/ProteinGym/zero_shot_substitutions_scores/PoET-2/'
dms_subs_dms_dir = "./PR/.cache/ProteinGym/DMS_ProteinGym_substitutions/"
dms_indels_output_dir = './PR/.cache/ProteinGym/zero_shot_indels_scores/PoET-2/'
dms_indels_dms_dir = "./PR/.cache/ProteinGym/DMS_ProteinGym_indels/"

clinical_subs_output_dir = './PR/.cache/ProteinGym/zero_shot_clinical_substitutions_scores/PoET-2'
clinical_subs_dms_dir = "./PR/.cache/ProteinGym/clinical_ProteinGym_substitutions"
clinical_indels_output_dir = "./PR/.cache/ProteinGym/zero_shot_clinical_indels_scores/PoET-2"
clinical_indels_dms_dir = "./PR/.cache/ProteinGym/clinical_ProteinGym_indels"

In [37]:
model_dict = {
    "model_list_zero_shot_substitutions_DMS": {
        "PoET-2": {"input_score_name": "PoET-2","location":"PoET-2","directionality":1,"key":"mutated_sequence","model_type":"Structure & MSA"}
    },
    "model_list_zero_shot_indels_DMS": {
        "PoET-2": {"input_score_name": "PoET-2","location":"PoET-2","directionality":1,"key":"mutated_sequence","model_type":"Structure & MSA"}
    },
    "model_list_zero_shot_substitutions_clinical": {
        "PoET-2": {"input_score_name": "PoET-2","location":"PoET-2","directionality":1,"key":"mutated_sequence","model_type":"Structure & MSA"}
    },
    "model_list_zero_shot_indels_clinical": {
#         "PoET-2": {"input_score_name": "PoET-2","location":"PoET-2","directionality":1,"key":"mutated_sequence","model_type":"Structure & MSA"},
        "PoET": {"input_score_name": "PoET_score","location":"PoET","directionality":1,"key":"mutated_sequence","model_type":"Structure & MSA"}
    },
}

## Define the shared functions
Estimate runtime

In [3]:
def estimate_runtime_dms_subs(test_cost, test_time, unit="hours", ref_df=dms_subs_ref_df):
    ref_df["proxy_cost"] = ref_df["DMS_total_number_mutants"] * ref_df["seq_len"]
    ref_df[f"estimated_{unit}"] = ref_df["proxy_cost"] / test_cost * test_time
    ref_df = ref_df.reset_index()
    print(ref_df[["index","DMS_id",f"estimated_{unit}"]]
          .sort_values(f"estimated_{unit}", ascending=False)
          .head(20)
         )
    return ref_df


def estimate_runtime_dms_indels(test_cost, test_time, unit="min", ref_df=dms_indels_ref_df):
    ref_df["proxy_cost"] = ref_df["DMS_total_number_mutants"]
    ref_df[f"estimated_{unit}"] = ref_df["proxy_cost"] / test_cost * test_time
    ref_df = ref_df.reset_index()
    print(ref_df[["index","DMS_id",f"estimated_{unit}"]]
          .sort_values(f"estimated_{unit}", ascending=False)
          .head(20)
         )
    return ref_df


def estimate_runtime_clinical_subs(test_cost, test_time, unit="min", 
                                   ref_df=clinical_subs_ref_df, clinical_sub_dir=clinical_subs_dms_dir):

    def calculate_total_number_mutants(DMS_id, clinical_sub_dir=clinical_sub_dir):
        return len(pd.read_csv(f"{clinical_sub_dir}/{DMS_id}.csv", sep=","))
    ref_df["total_number_mutants"] = ref_df['DMS_id'].map(calculate_total_number_mutants)
    ref_df["proxy_cost"] = ref_df["total_number_mutants"]
    ref_df[f"estimated_{unit}"] = ref_df["proxy_cost"] / test_cost * test_time
    ref_df = ref_df.reset_index()
    print(ref_df[["index","DMS_id","total_number_mutants", f"estimated_{unit}"]]
          .sort_values(f"estimated_{unit}", ascending=False)
          .head(20)
         )
    return ref_df


def estimate_runtime_clinical_indels(test_cost, test_time, unit="min", 
                                     ref_df=clinical_indels_ref_df, clinical_indel_dir=clinical_indels_dms_dir):

    def calculate_total_number_mutants(DMS_id, clinical_indel_dir=clinical_indel_dir):
        return len(pd.read_csv(f"{clinical_indel_dir}/{DMS_id}.csv", sep=","))
    ref_df["total_number_mutants"] = ref_df['DMS_id'].map(calculate_total_number_mutants)
    ref_df["proxy_cost"] = ref_df["total_number_mutants"]
    ref_df[f"estimated_{unit}"] = ref_df["proxy_cost"] / test_cost * test_time
    ref_df = ref_df.reset_index()
    print(ref_df[["index","DMS_id","total_number_mutants", f"estimated_{unit}"]]
          .sort_values(f"estimated_{unit}", ascending=False)
          .head(20)
         )
    return ref_df

Check runs

In [4]:
def check_runs(ref_df, output_dir):
    num_finished = 0
    index_unfinished = []
    for index, row in ref_df.iterrows():
        score_csv = output_dir + '/' + row['DMS_id'] + '.csv'
        if exists(score_csv):
            num_finished += 1
        else:
            index_unfinished.append(index)         

    if num_finished > 0:
        print(output_dir, index_unfinished, f"{len(index_unfinished)} unfinished")
        print(output_dir, f"{num_finished} finished")

Calculate Spearman and AUROC

In [5]:
def calc_spearman(ref_df, output_dir, dms_dir, model_dict_key):
    list_DMS = ref_df['DMS_id']
    model_list = model_dict[model_dict_key].keys()
    
    DMS_spearman = {'DMS_ID': [], 'number of mutants': []}

    for model in model_list:
        if model not in DMS_spearman:
            DMS_spearman[model] = np.full(shape=len(list_DMS), fill_value=np.nan)
        else:
            print(f"Duplicate model name {model}")

    for i, DMS_id in enumerate(list_DMS):
        print(DMS_id)
        DMS_spearman['DMS_ID'].append(DMS_id)

        DMS_file = dms_dir + os.sep + ref_df.iloc[i]["DMS_filename"]

        try:
            DMS_score = pd.read_csv(DMS_file)
            DMS_spearman['number of mutants'].append(len(DMS_score))
    #         print("DMS has entries: ", len(DMS_score))
        except:
            print(f"No DMS scores for {DMS_id}")
            DMS_spearman['number of mutants'].append(0)
            continue

        for model in model_list:
            try:
                model_score = pd.read_csv(output_dir + os.sep + DMS_id + ".csv")[[model]]
#                 model_score = pd.read_csv(output_dir + os.sep + model_dict[model_dict_key][model]["location"] + os.sep + DMS_id + ".csv")[[model]]
    #             print("PoET-2 has entries:", len(model_score))
            except:
                print(f"Scoring file for {DMS_id} in {model} missing")
                DMS_spearman[model][i] = np.nan # this line can be taken out later
                continue

            assert len(DMS_score) == len(model_score)
            result = pd.concat([DMS_score, model_score], axis=1)

            if 'DMS_score' in result.columns:
                DMS_spearman[model][i] = spearmanr(result['DMS_score'], result[model_dict[model_dict_key][model]["input_score_name"]]*model_dict[model_dict_key][model]["directionality"]).statistic
            elif model == "Site_Independent" or model == "EVmutation":
                DMS_spearman[model][i] = spearmanr(result['normalized_fitness'], result[model_dict[model_dict_key][model]["input_score_name"]]*model_dict[model_dict_key][model]["directionality"]).statistic            
            else:
                if model == "EVE_ensemble":
                    result['evol_indices_ensemble'] = result.drop('mutant', axis=1).mean(axis=1)
                merged_df = pd.merge(result, DMS_score, on=model_dict[model_dict_key][model]['key'], how='inner')
                DMS_spearman[model][i] = spearmanr(merged_df['DMS_score'], merged_df[model_dict[model_dict_key][model]["input_score_name"]]*model_dict[model_dict_key][model]["directionality"]).statistic
            
    return DMS_spearman


def build_performance_all_DMS(dms_spearman, ref_df, model_dict_key):
    """
    Convert per-assay Spearman values into the ProteinGym
    `performance_all_DMS['Spearman']` table:

        index   = DMS_id            (one row per assay)
        columns = [<one col per model>, 'number_mutants',
                   'UniProt_ID', 'Selection Type']

    so it can be fed straight into the 3-level groupby-mean averaging.
    Only UniProt_ID and Selection Type metadata are attached
    (no MSA_Neff_L_category / Taxon).
    """
    model_list = model_dict[model_dict_key].keys()
    # 1. per-assay spearman -> DataFrame, normalise id/count column names
    df = pd.DataFrame(dms_spearman) if isinstance(dms_spearman, dict) else dms_spearman.copy()
    df = df.rename(columns={'DMS_ID': 'DMS_id',
                            'number of mutants': 'number_mutants'})

    # 2. attach metadata from the reference file
    meta = ref_df[['DMS_id', 'UniProt_ID', 'coarse_selection_type']].rename(
        columns={'coarse_selection_type': 'Selection Type'})
    df = df.merge(meta, on='DMS_id', how='left')

    # 3. replace the missing-model sentinel (-1) with NaN so it is skipped
    #    by mean(numeric_only=True) at every averaging level
    model_cols = [m for m in model_list if m in df.columns]
    df[model_cols] = df[model_cols].replace(-1, np.nan)

    # 4. index by assay, order columns to mirror performance_all_DMS['Spearman']
    df = df.set_index('DMS_id')
    return df[model_cols + ['number_mutants', 'UniProt_ID', 'Selection Type']]


def average_across_assays(perf):
    """3-level nested mean: assays -> protein -> function category -> overall."""
    perf = perf.drop(columns=['number_mutants'])                       # not a metric
    per_protein  = perf.groupby(['UniProt_ID', 'Selection Type']).mean(numeric_only=True)
    per_function = per_protein.groupby('Selection Type').mean(numeric_only=True)
    return per_function.mean(numeric_only=True)                        # one value per model

Substitute for sklearn.roc_auc_score, shared by all auroc calculation functions

In [6]:
def roc_auc_score(y_true, y_score):
    """
    AUROC = P(score(positive) > score(negative)), tie-corrected.
    Equivalent to sklearn.metrics.roc_auc_score for binary {0,1} labels.
    """
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)

    pos = (y_true == 1)
    n_pos = pos.sum()
    n_neg = (~pos).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan                       # AUROC undefined with one class

    ranks = rankdata(y_score)               # average ranks -> handles ties
    auc = (ranks[pos].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return auc

In [7]:
def calc_auroc(ref_df, output_dir, clinical_dir, model_dict_key, label_col='DMS_bin_score'):
    """
    AUROC for clinical substitutions.
    `label_col` is the binary {0,1} ground-truth column in the clinical files
    (pathogenic = 1, benign = 0) -- set it to whatever your files use.
    """
    list_clin = ref_df['DMS_id']
    model_list = model_dict[model_dict_key].keys()

    clin_auroc = {'DMS_ID': [], 'number of variants': []}
    for model in model_list:
        if model not in clin_auroc:
            clin_auroc[model] = np.full(shape=len(list_clin), fill_value=np.nan)
        else:
            print(f"Duplicate model name {model}")

    for i, DMS_id in enumerate(list_clin):
        clin_auroc['DMS_ID'].append(DMS_id)

        clin_file = clinical_dir + os.sep + ref_df.iloc[i]["DMS_filename"]
        try:
            clin_labels = pd.read_csv(clin_file)
            clin_labels[label_col] = clin_labels[label_col].map({'Pathogenic': 0, 'Benign': 1})
            clin_auroc['number of variants'].append(len(clin_labels))
        except:
            print(f"No clinical labels for {DMS_id}")
            clin_auroc['number of variants'].append(0)
            continue

        for model in model_list:
            cfg = model_dict[model_dict_key][model]
            try:
                model_score = pd.read_csv(
                    output_dir + os.sep + DMS_id + ".csv")
            except:
                print(f"Scoring file for {DMS_id} in {model} missing")
                continue   # leave as NaN (not -1) so it is skipped in averaging

            # --- align labels with scores ---
            assert len(clin_labels) == len(model_score)
            merged = pd.concat([clin_labels.reset_index(drop=True),
                                model_score.reset_index(drop=True)], axis=1)
#             if cfg.get("key"):                       # per-mutant merge (e.g. EVE, EVmutation)
            if model == "EVE_ensemble":
                score_cols = [c for c in model_score.columns if c != cfg["key"]]
                model_score[cfg["input_score_name"]] = model_score[score_cols].mean(axis=1)
                merged = pd.merge(clin_labels, model_score, on=cfg["key"], how="inner")

            y_true  = merged[label_col]
            y_score = merged[cfg["input_score_name"]] * cfg["directionality"]

            # AUROC is undefined unless both classes are present and scores are finite
            valid = y_score.notna()
            if y_true[valid].nunique() < 2:
                print(f"Only one class for {DMS_id} in {model}; AUROC undefined")
                continue
            try:
                clin_auroc[model][i] = roc_auc_score(y_true=y_true[valid],
                                                     y_score=y_score[valid])
            except Exception as e:
                print(f"AUROC issue for {DMS_id} in {model}: {e}")

    return clin_auroc


def overall_auroc(clin_auroc, model_list, round_per_gene=True):
    """Official clinical-substitution aggregation: unweighted mean of per-gene AUCs."""
    df = pd.DataFrame(clin_auroc) if isinstance(clin_auroc, dict) else clin_auroc.copy()
    out = {}
    for model in model_list:
        aucs = df[model].dropna()                 # genes where AUC was computable
        if round_per_gene:
            aucs = aucs.round(3)                  # official rounds each gene first
        out[model] = round(aucs.mean(), 3) if len(aucs) else np.nan
    return pd.Series(out).sort_values(ascending=False)

Separately for substitutions (aggregate over genes) and indels (number of variants too small to aggregate meaningfully)

In [69]:
def calc_auroc_indels_pooled(ref_df, output_dir, clinical_dir, model_dict_key,
                             label_col='DMS_score_bin'):
    list_clin  = ref_df['DMS_id']
    model_list = model_dict[model_dict_key].keys()
    pooled = {m: {'y_true': [], 'y_score': []} for m in model_list}

    for i, DMS_id in enumerate(list_clin):
        clin_file = clinical_dir + os.sep + ref_df.iloc[i]["DMS_filename"]
        try:
            clin_labels = pd.read_csv(clin_file)
        except:
            print(f"No clinical labels for {DMS_id}"); continue

        # map + flip: Benign = 1 (same orientation as the official)
        clin_labels[label_col] = clin_labels[label_col].map({'Pathogenic': 0, 'Benign': 1})

        for model in model_list:
            cfg = model_dict[model_dict_key][model]
            try:
                model_score = pd.read_csv(
                    output_dir + os.sep + DMS_id + ".csv")
            except:
                print(f"Scoring file for {DMS_id} in {model} missing"); continue

#             if cfg.get("key"):
#                 merged = pd.merge(clin_labels, model_score, on=cfg["key"], how="inner")
#             else:
            if len(clin_labels) != len(model_score):
                print(DMS_id)
                print(f"clinical labels has {len(clin_labels)} entries; model scores has {len(model_score)}")
                continue
            assert len(clin_labels) == len(model_score)
            merged = pd.concat([clin_labels.reset_index(drop=True),
                                model_score.reset_index(drop=True)], axis=1)

            pooled[model]['y_true'].append(merged[label_col].values)
            pooled[model]['y_score'].append(
                (merged[cfg["input_score_name"]] * cfg["directionality"]).values)

    # one AUROC per model over the full pool
    out = {}
    for model in model_list:
        if not pooled[model]['y_true']:
            out[model] = np.nan; continue
        y_true  = np.concatenate(pooled[model]['y_true'])
        y_score = np.concatenate(pooled[model]['y_score'])
        valid = ~np.isnan(y_score)
        print(len(y_true), len(y_score))
        print(len(valid))
        if len(np.unique(y_true[valid])) < 2:
            out[model] = np.nan; continue
        out[model] = round(roc_auc_score(y_true[valid], y_score[valid]), 3)

    return pd.Series(out).sort_values(ascending=False)

Use the one below for clinical indels (the one above has issues)

In [9]:
def compute_pooled_auc(gene_ids, model_dict_key, scores_folder, clinical_folder, label_col="DMS_score_bin"):
    """Pool all variants across genes, then compute a single AUC per model.
    Used for indels where per-gene sample sizes are too small.
    """
    pooled = []
    model_config = model_dict[model_dict_key]
    model_list = model_dict[model_dict_key].keys()
    clin_auroc = dict()
    for gene_id in gene_ids:
        score_file = os.path.join(scores_folder, f"{gene_id}.csv")
        if not os.path.exists(score_file):
            print(f"Missing score file for {gene_id}")
            continue

        score_df = pd.read_csv(score_file)
        if "PoET-2" not in score_df.columns:
            print(f"Missing score column for {gene_id}")
            continue
        
        clinical_file = os.path.join(clinical_folder, f"{gene_id}.csv")
        if not os.path.exists(clinical_file):
            print(f"Missing clinical file for {gene_id}")
            continue

        true_df = pd.read_csv(clinical_file)
        if label_col not in true_df.columns:
            print(f"Missing {label_col} column for {gene_id} in clinical file")
            continue
        
        true_df[label_col] = true_df[label_col].map({'Pathogenic': 0, 'Benign': 1})
        
        assert len(true_df) == len(score_df), f"length mismatch for {gene_id}"
        merged = pd.concat([true_df[[label_col]].reset_index(drop=True),
                            score_df[['PoET-2']].reset_index(drop=True)], axis=1)
        pooled.append(merged)

    if not pooled:
        print("No data loaded!")
        return pd.Series(dtype=float)
    
    pooled = pd.concat(pooled, ignore_index=True)
    print(f"Pooled {len(pooled)} variants across {len(gene_ids)} genes")
    
    out = dict()
    for model in model_list:
        cfg = model_dict[model_dict_key][model]
        score_name = cfg["input_score_name"]
        if score_name not in pooled.columns:
            print(f"Missing score column '{score_name}' for {model}")
            out[model] = np.nan; continue

        y_true  = pooled[label_col]
        y_score = pooled[score_name] * cfg["directionality"]

        # AUROC is undefined unless both classes are present and scores are finite
        valid = y_score.notna() & y_true.notna()
        if y_true[valid].nunique() < 2:
            print(f"Only one class in {model}; AUROC undefined")
            out[model] = np.nan
            continue
        try:
            out[model] = round(roc_auc_score(y_true=y_true[valid].values,
                                             y_score=y_score[valid].values), 3)
        except Exception as e:
            print(f"AUROC issue in {model}: {e}")
            out[model] = np.nan
                               
    return pd.Series(out).sort_values(ascending=False)

# DMS subs

## Estimate runtime

In [27]:
zika_cost = 32778648
zika_runtime_hours = 3.0
dms_subs_df = estimate_runtime_dms_subs(zika_cost, zika_runtime_hours)

     index                            DMS_id  estimated_hours
181    181             SPG1_STRSG_Olson_2014        22.016678
73      73         HIS7_YEAST_Pokusaeva_2019         9.989748
182    182                SPG1_STRSG_Wu_2016         6.124104
128    128      POLG_CXB3N_Mattenberger_2021         3.141850
0        0   A0A140D2T1_ZIKV_Sourisseau_2019         3.000000
28      28            CAPSD_AAV2S_Sinai_2021         2.847379
122    122           PABP_YEAST_Melamed_2013         1.991313
124    124              PHOT_CHLRE_Chen_2023         1.809265
97      97               MSH2_HUMAN_Jia_2020         1.431746
129    129       POLG_DEN26_Suphatrakul_2023         1.391818
74      74             HMDH_HUMAN_Jiang_2019         1.369684
69      69             GRB2_HUMAN_Faure_2021         1.258480
67      67          GFP_AEQVI_Sarkisyan_2016         1.126459
6        6               A4_HUMAN_Seuma_2022         1.043771
31      31  CAS9_STRP1_Spencer_2017_positive         1.016276
55      

In [28]:
len(dms_subs_df)

217

## Check runs finished

In [44]:
check_runs(dms_subs_ref_df, dms_subs_output_dir)

217
./PR/.cache/ProteinGym/zero_shot_substitutions_scores/PoET-2/ []
./PR/.cache/ProteinGym/zero_shot_substitutions_scores/PoET-2/ 217


## Calculate Spearman

In [83]:
dms_subs_spearman = calc_spearman(dms_subs_ref_df, dms_subs_output_dir, dms_subs_dms_dir, "model_list_zero_shot_substitutions_DMS")

A0A140D2T1_ZIKV_Sourisseau_2019
A0A192B1T2_9HIV1_Haddox_2018
A0A1I9GEU1_NEIME_Kennouche_2019
A0A247D711_LISMN_Stadelmann_2021
A0A2Z5U3Z0_9INFA_Doud_2016
A0A2Z5U3Z0_9INFA_Wu_2014
A4_HUMAN_Seuma_2022
A4D664_9INFA_Soh_2019
A4GRB6_PSEAI_Chen_2020
AACC1_PSEAI_Dandage_2018
ACE2_HUMAN_Chan_2020
ADRB2_HUMAN_Jones_2020
AICDA_HUMAN_Gajula_2014_3cycles
AMFR_HUMAN_Tsuboyama_2023_4G3O
AMIE_PSEAE_Wrenbeck_2017
ANCSZ_Hobbs_2022
ARGR_ECOLI_Tsuboyama_2023_1AOY
B2L11_HUMAN_Dutta_2010_binding-Mcl-1
BBC1_YEAST_Tsuboyama_2023_1TG0
BCHB_CHLTE_Tsuboyama_2023_2KRU
BLAT_ECOLX_Deng_2012
BLAT_ECOLX_Firnberg_2014
BLAT_ECOLX_Jacquier_2013
BLAT_ECOLX_Stiffler_2015
BRCA1_HUMAN_Findlay_2018
BRCA2_HUMAN_Erwood_2022_HEK293T
C6KNH7_9INFA_Lee_2018
CALM1_HUMAN_Weile_2017
CAPSD_AAV2S_Sinai_2021
CAR11_HUMAN_Meitlis_2020_gof
CAR11_HUMAN_Meitlis_2020_lof
CAS9_STRP1_Spencer_2017_positive
CASP3_HUMAN_Roychowdhury_2020
CASP7_HUMAN_Roychowdhury_2020
CATR_CHLRE_Tsuboyama_2023_2AMI
CBPA2_HUMAN_Tsuboyama_2023_1O6X
CBS_HUMAN_Sun_2020

In [84]:
dms_subs_spearman.keys()

dict_keys(['DMS_ID', 'number of mutants', 'PoET-2'])

In [91]:
dms_subs_perf = build_performance_all_DMS(dms_subs_spearman, dms_subs_ref_df, "model_list_zero_shot_substitutions_DMS")
dms_subs_leaderboard = average_across_assays(dms_subs_perf).sort_values(ascending=False)
print(dms_subs_leaderboard)

PoET-2    0.500141
dtype: float64


In [86]:
len(dms_subs_spearman['DMS_ID'])

217

In [87]:
dms_subs_spearman['PoET-2']

array([0.4654173 , 0.49981055, 0.02942379, 0.54325031, 0.60054724,
       0.56368136, 0.40032339, 0.38857022, 0.73920917, 0.52508455,
       0.27909446, 0.5513496 , 0.45951355, 0.35285138, 0.6415395 ,
       0.61182726, 0.58639299, 0.40883849, 0.52611437, 0.51560449,
       0.5507582 , 0.76295805, 0.71496152, 0.7589067 , 0.57461511,
       0.43498884, 0.47255988, 0.24921442, 0.57057537, 0.34600688,
       0.51516666, 0.19380037, 0.63869412, 0.64962204, 0.67843703,
       0.79627836, 0.37571409, 0.74026488, 0.48397551, 0.55861699,
       0.35200566, 0.3129669 , 0.66120122, 0.71039203, 0.61378154,
       0.62157748, 0.68591186, 0.7238249 , 0.54254705, 0.47811423,
       0.81547185, 0.5217321 , 0.50983376, 0.46734614, 0.40440058,
       0.34207421, 0.20238294, 0.79981439, 0.57168708, 0.41810324,
       0.57056208, 0.41844762, 0.57066311, 0.40459642, 0.53877755,
       0.21403597, 0.47273213, 0.75377826, 0.40393965, 0.626085  ,
       0.67027183, 0.42354651, 0.4497878 , 0.59106765, 0.45831

Compare with authors' spearman

In [70]:
author_dir = './PR/proteingym_v1.3_zero_shot_substitutions_scores_poet_2_afdb_v6'

author_dms_subs_spearman = calc_spearman(dms_subs_ref_df, author_dir, dms_subs_dms_dir, "model_list_zero_shot_substitutions_DMS")

A0A140D2T1_ZIKV_Sourisseau_2019
A0A192B1T2_9HIV1_Haddox_2018
A0A1I9GEU1_NEIME_Kennouche_2019
A0A247D711_LISMN_Stadelmann_2021
A0A2Z5U3Z0_9INFA_Doud_2016
A0A2Z5U3Z0_9INFA_Wu_2014
A4_HUMAN_Seuma_2022
A4D664_9INFA_Soh_2019
A4GRB6_PSEAI_Chen_2020
AACC1_PSEAI_Dandage_2018
ACE2_HUMAN_Chan_2020
ADRB2_HUMAN_Jones_2020
AICDA_HUMAN_Gajula_2014_3cycles
AMFR_HUMAN_Tsuboyama_2023_4G3O
AMIE_PSEAE_Wrenbeck_2017
ANCSZ_Hobbs_2022
ARGR_ECOLI_Tsuboyama_2023_1AOY
B2L11_HUMAN_Dutta_2010_binding-Mcl-1
BBC1_YEAST_Tsuboyama_2023_1TG0
BCHB_CHLTE_Tsuboyama_2023_2KRU
BLAT_ECOLX_Deng_2012
BLAT_ECOLX_Firnberg_2014
BLAT_ECOLX_Jacquier_2013
BLAT_ECOLX_Stiffler_2015
BRCA1_HUMAN_Findlay_2018
BRCA2_HUMAN_Erwood_2022_HEK293T
C6KNH7_9INFA_Lee_2018
CALM1_HUMAN_Weile_2017
CAPSD_AAV2S_Sinai_2021
CAR11_HUMAN_Meitlis_2020_gof
CAR11_HUMAN_Meitlis_2020_lof
CAS9_STRP1_Spencer_2017_positive
CASP3_HUMAN_Roychowdhury_2020
CASP7_HUMAN_Roychowdhury_2020
CATR_CHLRE_Tsuboyama_2023_2AMI
CBPA2_HUMAN_Tsuboyama_2023_1O6X
CBS_HUMAN_Sun_2020

In [82]:
print(author_dms_subs_spearman['PoET-2'].mean())
print(author_dms_subs_spearman['PoET-2'])

0.5190723895302114
[0.46544408 0.49937867 0.02947372 0.54323114 0.6006638  0.56372733
 0.40038191 0.38856525 0.73909807 0.52508172 0.27919571 0.55136898
 0.45956223 0.35283168 0.6417437  0.61188734 0.58636293 0.40843302
 0.52595022 0.51549601 0.5506292  0.76282335 0.71493029 0.7588428
 0.57460455 0.43493919 0.47265746 0.24917966 0.57065009 0.34604249
 0.51530985 0.19368972 0.63865908 0.64957522 0.67845375 0.79628003
 0.37571131 0.74028498 0.48387946 0.55863634 0.35201846 0.31293045
 0.66095132 0.71029155 0.61371907 0.62165641 0.68592353 0.72386289
 0.54249975 0.47812616 0.81544373 0.5217904  0.50975557 0.46742387
 0.4043547  0.34179988 0.20233161 0.79983589 0.57160828 0.41815228
 0.57059175 0.4184206  0.57071078 0.40465943 0.53873959 0.21409544
 0.47278544 0.75378995 0.40382541 0.62614044 0.67031513 0.42361934
 0.44987616 0.59104739 0.45783158 0.36429206 0.36746876 0.4463608
 0.50346001 0.43794032 0.38857153 0.58680524 0.47695009 0.35878543
 0.07993494 0.56717038 0.43275332 0.36778351 

In [72]:
df_check = pd.DataFrame()
df_check['is_match'] = (dms_subs_spearman['PoET-2'] - author_dms_subs_spearman['PoET-2']) < 0.001

In [73]:
df_check

,is_match
0,True
1,True
2,True
3,True
4,True
...,...
212,True
213,True
214,True
215,True


In [74]:
df_check.sum()

is_match    216
dtype: int64

In [75]:
df_check[df_check['is_match'] == False]

,is_match
86,False


In [76]:
dms_subs_spearman['PoET-2'][86]

np.float64(0.4341378534463362)

In [77]:
author_dms_subs_spearman['PoET-2'][86]

np.float64(0.43275331883297086)

In [79]:
dms_subs_ref_df['DMS_id'][86]

'KCNH2_HUMAN_Kozek_2020'

**Conclusions:** `KCNH2_HUMAN_Kozek_2020` has assay Spearman difference over 0.001; everything else has Spearman difference less than 0.001

# DMS indels
## Estimate runtime

In [15]:
BLAT_ECOLX_Gonzalez_runtime_min = 7.5
BLAT_ECOLX_Gonzalez_cost = 4751
dms_indel_df = estimate_runtime_dms_indels(BLAT_ECOLX_Gonzalez_cost, BLAT_ECOLX_Gonzalez_runtime_min)

    index                                  DMS_id  estimated_min
7       7  CAPSD_AAV2S_Sinai_2021_designed_indels     356.763839
8       8   CAPSD_AAV2S_Sinai_2021_library_indels      39.320143
23     23       KCNJ2_MOUSE_Macdonald_2022_indels      16.577036
21     21        HIS7_YEAST_Pokusaeva_2019_indels       9.632709
6       6         BLAT_ECOLX_Gonzalez_2019_indels       7.500000
3       3           B1LPA6_ECOSM_Russ_2020_indels       4.852663
0       0              A4_HUMAN_Seuma_2022_indels       3.703431
48     48    S22A1_HUMAN_Yee_2023_activity_indels       0.773521
47     47   S22A1_HUMAN_Yee_2023_abundance_indels       0.678804
33     33            P53_HUMAN_Kotler_2018_indels       0.538308
41     41       Q8EG35_SHEON_Campbell_2022_indels       0.522522
40     40          PTEN_HUMAN_Mighell_2018_indels       0.495685
10     10  CBPA2_HUMAN_Tsuboyama_2023_1O6X_indels       0.323616
9       9   CATR_CHLRE_Tsuboyama_2023_2AMI_indels       0.310987
12     12   CSN4_MOUSE_Ts

In [16]:
dms_indel_df[dms_indel_df['estimated_min'] < 5]["estimated_min"].sum()

np.float64(23.595558829720062)

## Check runs finished

In [45]:
check_runs(dms_indels_ref_df, dms_indels_output_dir)

66
./PR/.cache/ProteinGym/zero_shot_indels_scores/PoET-2/ []
./PR/.cache/ProteinGym/zero_shot_indels_scores/PoET-2/ 66


## Calculate Spearman

In [88]:
dms_indels_spearman = calc_spearman(dms_indels_ref_df, dms_indels_output_dir, dms_indels_dms_dir, "model_list_zero_shot_indels_DMS")

A4_HUMAN_Seuma_2022_indels
AMFR_HUMAN_Tsuboyama_2023_4G3O_indels
ARGR_ECOLI_Tsuboyama_2023_1AOY_indels
B1LPA6_ECOSM_Russ_2020_indels
BBC1_YEAST_Tsuboyama_2023_1TG0_indels
BCHB_CHLTE_Tsuboyama_2023_2KRU_indels
BLAT_ECOLX_Gonzalez_2019_indels
CAPSD_AAV2S_Sinai_2021_designed_indels
CAPSD_AAV2S_Sinai_2021_library_indels
CATR_CHLRE_Tsuboyama_2023_2AMI_indels
CBPA2_HUMAN_Tsuboyama_2023_1O6X_indels
CBX4_HUMAN_Tsuboyama_2023_2K28_indels
CSN4_MOUSE_Tsuboyama_2023_1UFM_indels
CUE1_YEAST_Tsuboyama_2023_2MYX_indels
DN7A_SACS2_Tsuboyama_2023_1JIC_indels
DNJA1_HUMAN_Tsuboyama_2023_2LO1_indels
DOCK1_MOUSE_Tsuboyama_2023_2M0Y_indels
EPHB2_HUMAN_Tsuboyama_2023_1F0M_indels
FECA_ECOLI_Tsuboyama_2023_2D1U_indels
HCP_LAMBD_Tsuboyama_2023_2L6Q_indels
HECD1_HUMAN_Tsuboyama_2023_3DKM_indels
HIS7_YEAST_Pokusaeva_2019_indels
ILF3_HUMAN_Tsuboyama_2023_2L33_indels
KCNJ2_MOUSE_Macdonald_2022_indels
MAFG_MOUSE_Tsuboyama_2023_1K1V_indels
MBD11_ARATH_Tsuboyama_2023_6ACV_indels
MYO3_YEAST_Tsuboyama_2023_2BTT_indels
NK

In [89]:
dms_indels_spearman['PoET-2'].mean()

np.float64(0.6045118056439708)

In [92]:
dms_indels_perf = build_performance_all_DMS(dms_indels_spearman, dms_indels_ref_df, "model_list_zero_shot_indels_DMS")
dms_indels_leaderboard = average_across_assays(dms_indels_perf).sort_values(ascending=False)
print(dms_indels_leaderboard)

PoET-2    0.573833
dtype: float64


Compare with authors' raw scores

In [93]:
author_dir = './PR/proteingym_v1.3_zero_shot_indels_scores_poet_2_afdb_v6'

author_dms_indels_spearman = calc_spearman(dms_indels_ref_df, author_dir, dms_indels_dms_dir, "model_list_zero_shot_indels_DMS")

A4_HUMAN_Seuma_2022_indels
AMFR_HUMAN_Tsuboyama_2023_4G3O_indels
ARGR_ECOLI_Tsuboyama_2023_1AOY_indels
B1LPA6_ECOSM_Russ_2020_indels
BBC1_YEAST_Tsuboyama_2023_1TG0_indels
BCHB_CHLTE_Tsuboyama_2023_2KRU_indels
BLAT_ECOLX_Gonzalez_2019_indels
CAPSD_AAV2S_Sinai_2021_designed_indels
CAPSD_AAV2S_Sinai_2021_library_indels
CATR_CHLRE_Tsuboyama_2023_2AMI_indels
CBPA2_HUMAN_Tsuboyama_2023_1O6X_indels
CBX4_HUMAN_Tsuboyama_2023_2K28_indels
CSN4_MOUSE_Tsuboyama_2023_1UFM_indels
CUE1_YEAST_Tsuboyama_2023_2MYX_indels
DN7A_SACS2_Tsuboyama_2023_1JIC_indels
DNJA1_HUMAN_Tsuboyama_2023_2LO1_indels
DOCK1_MOUSE_Tsuboyama_2023_2M0Y_indels
EPHB2_HUMAN_Tsuboyama_2023_1F0M_indels
FECA_ECOLI_Tsuboyama_2023_2D1U_indels
HCP_LAMBD_Tsuboyama_2023_2L6Q_indels
HECD1_HUMAN_Tsuboyama_2023_3DKM_indels
HIS7_YEAST_Pokusaeva_2019_indels
ILF3_HUMAN_Tsuboyama_2023_2L33_indels
KCNJ2_MOUSE_Macdonald_2022_indels
MAFG_MOUSE_Tsuboyama_2023_1K1V_indels
MBD11_ARATH_Tsuboyama_2023_6ACV_indels
MYO3_YEAST_Tsuboyama_2023_2BTT_indels
NK

In [94]:
print(author_dms_indels_spearman['PoET-2'].mean())
print(author_dms_indels_spearman['PoET-2'])

0.6045033947330347
[0.56150762 0.67250371 0.58878433 0.47079534 0.67398596 0.552334
 0.64937529 0.67992963 0.2942418  0.55163256 0.59350476 0.58838327
 0.33550632 0.71003914 0.57251085 0.48090549 0.67834594 0.63335734
 0.56527763 0.71936832 0.64974658 0.7101096  0.68104001 0.49884478
 0.00143612 0.74802221 0.67674637 0.64818194 0.58357674 0.46798231
 0.89577843 0.82308048 0.6206338  0.52770156 0.81645968 0.37778927
 0.83824355 0.77500091 0.71418701 0.41959695 0.78400153 0.4892994
 0.3380365  0.37594335 0.74665602 0.62541591 0.31824345 0.50739422
 0.56398312 0.79568848 0.80709247 0.5707256  0.50752766 0.80244298
 0.69077747 0.69577602 0.84040349 0.75448361 0.79939269 0.26467708
 0.47252718 0.67928119 0.61051835 0.61890281 0.68163171 0.50995412]


In [95]:
df_check = pd.DataFrame()
df_check['is_match'] = (dms_indels_spearman['PoET-2'] - author_dms_indels_spearman['PoET-2']) < 0.001
df_check.sum()

is_match    66
dtype: int64

In [96]:
len(df_check)

66

In [97]:
author_dms_indels_perf = build_performance_all_DMS(author_dms_indels_spearman, dms_indels_ref_df, "model_list_zero_shot_indels_DMS")
author_dms_indels_leaderboard = average_across_assays(author_dms_indels_perf).sort_values(ascending=False)
print(dms_indels_leaderboard)

PoET-2    0.573833
dtype: float64


**Conclusion:** The authors reported 0.573, so update it

# Clinical subs

## Estimate run time

In [32]:
test_runtime_min = 4
test_cost = 26
clinical_subs_df = estimate_runtime_clinical_subs(test_cost, test_runtime_min)

      index          DMS_id  total_number_mutants  estimated_min
634     634  NP_001159435.1                   741     114.000000
318     318     NP_000518.1                   639      98.307692
301     301     NP_000486.1                   508      78.153846
1563   1563     NP_009225.1                   456      70.153846
223     223     NP_000341.2                   448      68.923077
876     876     NP_001835.3                   402      61.846154
178     178     NP_000268.1                   399      61.384615
334     334     NP_000539.2                   376      57.846154
49       49     NP_000080.2                   344      52.923077
48       48     NP_000079.2                   338      52.000000
163     163     NP_000248.2                   301      46.307692
50       50     NP_000081.2                   290      44.615385
157     157     NP_000242.1                   288      44.307692
332     332     NP_000537.3                   271      41.692308
2395   2395     NP_742105

In [33]:
len(clinical_subs_df)

2525

In [16]:
clinical_subs_df[["index","DMS_id","total_number_mutants", "estimated_min"]].to_csv("./PR/clinical_subs_estimated_time.csv")

## Check run finished

In [7]:
check_runs(clinical_subs_ref_df, clinical_subs_output_dir)

./PR/.cache/ProteinGym/zero_shot_clinical_substitutions_scores/PoET-2 [] 0 unfinished
./PR/.cache/ProteinGym/zero_shot_clinical_substitutions_scores/PoET-2 2525 finished


## Calculate AUROC

In [49]:
clinical_subs_auroc = calc_auroc(clinical_subs_ref_df, clinical_subs_output_dir, clinical_subs_dms_dir, "model_list_zero_shot_substitutions_clinical")

In [50]:
overall_auroc(clinical_subs_auroc, list(model_dict["model_list_zero_shot_substitutions_clinical"].keys()))

PoET-2    0.932
dtype: float64

# clinical indels
## Estimate runtime

In [40]:
test_runtime_min = 4
test_cost = 26
clinical_indels_df = estimate_runtime_clinical_indels(test_cost, test_runtime_min)

      index          DMS_id  total_number_mutants  estimated_min
178     178     NP_000518.1                    31       4.769231
236     236  NP_001035957.1                    28       4.307692
275     275  NP_001159435.1                    21       3.230769
217     217  NP_001009944.3                    18       2.769231
48       48     NP_000129.3                    16       2.461538
83       83     NP_000240.1                    14       2.153846
265     265  NP_001123910.1                    13       2.000000
369     369     NP_001835.3                    12       1.846154
88       88     NP_000248.2                    12       1.846154
784     784     NP_597677.2                    11       1.692308
191     191     NP_000539.2                    10       1.538462
116     116     NP_000313.2                    10       1.538462
36       36     NP_000086.2                     9       1.384615
811     811     NP_733821.1                     9       1.384615
530     530     NP_005219

In [41]:
clinical_indels_df["estimated_min"].sum()

np.float64(397.07692307692304)

In [42]:
len(clinical_indels_df)

1555

## Check run finished

In [18]:
check_runs(clinical_indels_ref_df, clinical_indels_output_dir)

./PR/.cache/ProteinGym/zero_shot_clinical_indels_scores/PoET-2 [] 0 unfinished
./PR/.cache/ProteinGym/zero_shot_clinical_indels_scores/PoET-2 1555 finished


In [19]:
clinical_indels_ref_df

,DMS_id,DMS_filename,target_seq,mutated_sequence_column,MSA_start,MSA_end,MSA_filename,weight_file_name,dataset,total_number_mutants,proxy_cost,estimated_min
0,NP_000007.1,NP_000007.1.csv,MAAGFGRCCRVLRSISRFHWRSQHTKANRQREPGLGFSFEFTEQQK...,mutated_sequence,1.0,421.0,NP_000007.1.a2m,NP_000007.1.npy,ClinVar,2,2,4.50
1,NP_000008.1,NP_000008.1.csv,MAAALLARASGPARRALCPRAWRQLHTIYQSVELPETHQMLLQTCR...,mutated_sequence,1.0,412.0,NP_000008.1.a2m,NP_000008.1.npy,ClinVar,1,1,2.25
2,NP_000009.1,NP_000009.1.csv,MQAARMAASLGRQLLRLGGGSSRLTALLGQPRPGPARRPYAGGAAQ...,mutated_sequence,1.0,655.0,NP_000009.1.a2m,NP_000009.1.npy,ClinVar,4,4,9.00
3,NP_000010.1,NP_000010.1.csv,MAVLAALLRSGARSRSPLLRRLVQEIRYVERSYVSKPTLKEVVIVS...,mutated_sequence,1.0,427.0,NP_000010.1.a2m,NP_000010.1.npy,ClinVar,4,4,9.00
4,NP_000011.2,NP_000011.2.csv,MTLGSPRKGLLMLLMALVTQGDPVKPSRGPLVTCTCESPHCKGPTC...,mutated_sequence,1.0,503.0,NP_000011.2.a2m,NP_000011.2.npy,ClinVar,6,6,13.50
...,...,...,...,...,...,...,...,...,...,...,...,...
1550,UPI000008AB1D,UPI000008AB1D.csv,MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,mutated_sequence,1.0,365.0,UPI000008AB1D.a2m,UPI000008AB1D.npy,gnomAD,1,1,2.25
1551,UPI0000456761,UPI0000456761.csv,MSAPPVLRPPSPLLPVAAAAAAAAAALVPGSGPGPAPFLAPVAAPV...,mutated_sequence,1.0,912.0,UPI0000456761.a2m,UPI0000456761.npy,gnomAD,1,1,2.25
1552,UPI000015FEE1,UPI000015FEE1.csv,MVFESVVVDVLNRFLGDYVVDLDTSQLSLGIWKGAVALKNLQIKEN...,mutated_sequence,1.0,3095.0,UPI000015FEE1.a2m,UPI000015FEE1.npy,gnomAD,1,1,2.25
1553,UPI000012CBC3,UPI000012CBC3.csv,MKALIAALLLITLQYSCAVSPTDCSAVEPEAEKALDLINKRRRDGY...,mutated_sequence,1.0,525.0,UPI000012CBC3.a2m,UPI000012CBC3.npy,gnomAD,1,1,2.25


In [30]:
def calculate_total_number_mutants(DMS_id, clinical_indel_dir=clinical_indels_dms_dir):
    return len(pd.read_csv(f"{clinical_indel_dir}/{DMS_id}.csv", sep=","))
clinical_indels_ref_df["total_number_mutants"] = clinical_indels_ref_df['DMS_id'].map(calculate_total_number_mutants)

In [11]:
clinical_indels_ref_df["total_number_mutants"].iloc[:81].sum()

np.int64(241)

In [14]:
clinical_indels_ref_df["total_number_mutants"].iloc[778:868].sum()

np.int64(160)

In [10]:
test_cost = 160
test_time = 360
clinical_indels_df = estimate_runtime_clinical_indels(test_cost, test_time, unit="min")

      index          DMS_id  total_number_mutants  estimated_min
178     178     NP_000518.1                    31          69.75
236     236  NP_001035957.1                    28          63.00
275     275  NP_001159435.1                    21          47.25
217     217  NP_001009944.3                    18          40.50
48       48     NP_000129.3                    16          36.00
83       83     NP_000240.1                    14          31.50
265     265  NP_001123910.1                    13          29.25
369     369     NP_001835.3                    12          27.00
88       88     NP_000248.2                    12          27.00
784     784     NP_597677.2                    11          24.75
191     191     NP_000539.2                    10          22.50
116     116     NP_000313.2                    10          22.50
36       36     NP_000086.2                     9          20.25
811     811     NP_733821.1                     9          20.25
530     530     NP_005219

In [16]:
clinical_indels_df[["index","DMS_id","total_number_mutants", "estimated_min"]].to_csv("./PR/clinical_indels_estimated_time.csv")

## Calculate AUROC

In [57]:
clinical_indels_auroc = calc_auroc_indels_pooled(clinical_indels_ref_df, clinical_indels_output_dir, clinical_indels_dms_dir, "model_list_zero_shot_indels_clinical")

2878 2878
2878


In [58]:
clinical_indels_auroc

PoET-2    0.784
dtype: float64

In [60]:
author_dir = './PR/proteingym_v1.3_zero_shot_clinical_indels_scores_poet_2'

author_clinical_indels_auroc = calc_auroc_indels_pooled(clinical_indels_ref_df, author_dir, clinical_indels_dms_dir, "model_list_zero_shot_indels_clinical")

2878 2878
2878


In [61]:
author_clinical_indels_auroc

PoET-2    0.784
dtype: float64

In [114]:
gene_ids = clinical_indels_ref_df['DMS_id'].tolist()
compute_pooled_auc(gene_ids, "model_list_zero_shot_indels_clinical", clinical_indels_output_dir, clinical_indels_dms_dir)

Pooled 2878 variants across 1555 genes


PoET-2    0.934
dtype: float64

In [115]:
compute_pooled_auc(gene_ids, "model_list_zero_shot_indels_clinical", author_dir, clinical_indels_dms_dir)

Pooled 2878 variants across 1555 genes


PoET-2    0.934
dtype: float64

To figure out whether this is correct, apply the function to PoET

In [47]:
def compute_pooled_auc_test(gene_ids, model_dict_key, scores_folder, clinical_folder, label_col="DMS_score_bin"):
    """Pool all variants across genes, then compute a single AUC per model.
    Used for indels where per-gene sample sizes are too small.
    """
    pooled = []
    model_config = model_dict[model_dict_key]
    model_list = model_dict[model_dict_key].keys()
    clin_auroc = dict()
    for gene_id in gene_ids:
        score_file = os.path.join(scores_folder, f"{gene_id}.csv")
        if not os.path.exists(score_file):
            print(f"Missing score file for {gene_id}")
            continue

        score_df = pd.read_csv(score_file)
        if "PoET_score" not in score_df.columns:
            print(f"Missing score column for {gene_id}")
            continue
        
        clinical_file = os.path.join(clinical_folder, f"{gene_id}.csv")
        if not os.path.exists(clinical_file):
            print(f"Missing clinical file for {gene_id}")
            continue

        true_df = pd.read_csv(clinical_file)
        if label_col not in true_df.columns:
            print(f"Missing {label_col} column for {gene_id} in clinical file")
            continue
        
        true_df[label_col] = true_df[label_col].map({'Pathogenic': 0, 'Benign': 1})
        
        assert len(true_df) == len(score_df), f"length mismatch for {gene_id}"
#         merged = pd.concat([true_df[[label_col]].reset_index(drop=True),
#                             score_df[['PoET_score']].reset_index(drop=True)], axis=1)
        merged = pd.merge(true_df, score_df, on='mutated_sequence')
        pooled.append(merged[["mutated_sequence", label_col, "PoET_score"]])

    if not pooled:
        print("No data loaded!")
        return pd.Series(dtype=float)
    
    pooled = pd.concat(pooled, ignore_index=True)
    print(f"Pooled {len(pooled)} variants across {len(gene_ids)} genes")
    
    out = dict()
    for model in model_list:
        cfg = model_dict[model_dict_key][model]
        score_name = cfg["input_score_name"]
        if score_name not in pooled.columns:
            print(f"Missing score column '{score_name}' for {model}")
            out[model] = np.nan; continue

        y_true  = pooled[label_col]
        y_score = pooled[score_name] * cfg["directionality"]

        # AUROC is undefined unless both classes are present and scores are finite
        valid = y_score.notna() & y_true.notna()
        if y_true[valid].nunique() < 2:
            print(f"Only one class in {model}; AUROC undefined")
            out[model] = np.nan
            continue
        try:
            out[model] = round(roc_auc_score(y_true=y_true[valid].values,
                                             y_score=y_score[valid].values), 3)
        except Exception as e:
            print(f"AUROC issue in {model}: {e}")
            out[model] = np.nan
                               
    return pd.Series(out).sort_values(ascending=False)

In [48]:
gene_ids = clinical_indels_ref_df['DMS_id'].tolist()
compute_pooled_auc_test(gene_ids, "model_list_zero_shot_indels_clinical", "PR/.cache/ProteinGym/v1.2/zero_shot_clinical_indels_scores/PoET", clinical_indels_dms_dir)

Pooled 2878 variants across 1555 genes


PoET    0.92
dtype: float64